In [1]:
import os
import json
import re
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

client = OpenAI(api_key=OPENAI_API_KEY)

In [2]:
# 최신 동향 서치

# 1. 공신력 있는 사이트 한정
# -------------------------
ALL_DOMAINS = [
    # 공공 / 연구
    "go.kr",
    "or.kr",
    "ac.kr",
    "korea.kr",
    "msit.go.kr",
    "kisdi.re.kr",
    "nia.or.kr",

    # 뉴스
    "chosun.com",
    "joongang.co.kr",
    "donga.com",
    "hani.co.kr",
    "mk.co.kr",
    "mt.co.kr",
    "edaily.co.kr",
    "etnews.com",
    "zdnet.co.kr",
    "electimes.com",
    "digitaltoday.co.kr"
]

def web_search_trusted(query):
    site_filter = " OR ".join([f"site:{d}" for d in ALL_DOMAINS])
    search_query = f"{query} ({site_filter})"

    response = client.responses.create(
        model = "gpt-5.4",
        tools=[{"type": "web_search"}],
        input=search_query
    )

    return response.output_text

In [3]:
# 2. URL 추출
# -------------------------
def extract_urls(text, k=10):
    urls = re.findall(r'https?://[^\s)]+', text)
    urls = list(set(urls))
    return urls[:k]

In [4]:
# 3. 요약 
# -------------------------
def summarize_results(web_text, max_lines=10):

    summary_prompt = f"""
    아래 내용을 기반으로 통신 산업 최신 동향을 {max_lines}줄로 정리하라.
    반드시:
    - 한 줄당 하나의 핵심 내용
    - 짧고 명확하게
    - bullet 없이 텍스트만

    내용:
    {web_text}
    """

    response = client.responses.create(
        model="gpt-5.4",
        input=summary_prompt
    )

    lines = [line.strip() for line in response.output_text.split("\n") if line.strip()]

    return lines[:max_lines]

In [5]:
# 4. 최종 함수
# -------------------------
def get_telco_trend_with_news(query="통신 산업 최신 동향 2026", k=10):
    web_text = web_search_trusted(query)

    summary = summarize_results(web_text, max_lines=k)
    urls = extract_urls(web_text, k=k)

    return summary, urls

In [6]:
summary, urls = get_telco_trend_with_news()

print("통신 산업 최신 동향 (요약)")
for line in summary:
    print("-", line)

print("\n관련 뉴스 URL")
for i, url in enumerate(urls, 1):
    print(f"{i}. {url}")

통신 산업 최신 동향 (요약)
- 2026년 한국 통신 산업은 전통 통신에서 AI 인프라 산업으로 재편되고 있다.
- 통신 3사는 수익 중심을 AI·클라우드·B2B 사업으로 옮기고 있다.
- 기업·공공 고객 대상 AI 서비스와 디지털 전환 사업이 빠르게 확대되고 있다.
- AI 네이티브 네트워크가 차세대 통신의 핵심 기술로 부상하고 있다.
- 네트워크는 자율형 운영과 AI 기반 최적화가 가능한 지능형 인프라로 진화하고 있다.
- 경쟁 기준은 단순 통신 품질에서 AI 운영 역량과 서비스 융합력으로 이동하고 있다.
- 6G와 위성통신 융합이 본격화되며 비지상망 경쟁이 중요해지고 있다.
- 초저지연·초연결 서비스 구현을 위한 차세대 연결 기술 투자가 늘고 있다.
- 보안과 신뢰성은 통신사의 핵심 경쟁력으로 자리 잡고 있다.
- 정부와 공공 부문도 AI 네트워크와 차세대 연구망 투자를 확대하고 있다.

관련 뉴스 URL
1. https://www.etnews.com/20260210000294?utm_source=openai
2. https://nia.or.kr/site/nia_kor/ex/bbs/ListBusiness.do?pageIndex=3319&utm_source=openai
3. https://www.etnews.com/20260302000017?utm_source=openai
4. https://www.nia.or.kr/site/nia_kor/ex/bbs/View.do?bcIdx=28932&cbIdx=25932&parentSeq=28932&utm_source=openai


In [ ]:
# 라우터
def is_telco_trend_query(user_input):
    telco_keywords = ["통신", "통신사", "5G", "인터넷", "telecom"]
    trend_keywords = ["동향", "현황", "트렌드", "뉴스", "최근", "최신"]

    return (
        any(k in user_input for k in telco_keywords) and
        any(t in user_input for t in trend_keywords)
    )

def route_user_query(user_input):

    if is_telco_trend_query(user_input):
        summary, urls = get_telco_trend_with_news(user_input)
        return summary, urls

    # 기본 LLM 응답
    return general_chat(user_input)